# Globus (ALCF) → IRI Token Bootstrap Notebook

This notebook performs an interactive OAuth2 flow using **Globus Auth** to authenticate with ALCF.

It will:
1. Start an OAuth authorization flow
2. Exchange the authorization code for an access token
3. Display expiration details
4. Save the "iri_api"-scoped token to `~/.iri_token.json` as `IRI_API_TOKEN`


## 1) Setup and Imports

In [ ]:
import sys
!{sys.executable} -m pip install globus-sdk python-dotenv

In [ ]:
import os
import datetime
import time
import json
import pprint
from pathlib import Path

import globus_sdk
from dotenv import load_dotenv


## 2) Configuration

In [ ]:
# Load environment variables from .env (if present)
load_dotenv()

APP_NAME = "alcf_facility_api_app"

CLIENT_ID = "8b84fc2d-49e9-49ea-b54d-b3a29a70cf31"

SCOPE_CLIENT_ID = "6be511f6-a071-471f-9bc0-02a0d0836723"

SCOPES = [f"https://auth.globus.org/scopes/{SCOPE_CLIENT_ID}/filesystem"]


## 3) Create OAuth Flow

In [ ]:
# Create a native client (NOT confidential)
client = globus_sdk.NativeAppAuthClient(CLIENT_ID)

# Start OAuth flow
client.oauth2_start_flow(
    requested_scopes=SCOPES,
    refresh_tokens=True
)

# Get authorization URL
authorize_url = client.oauth2_get_authorize_url()
print("\n👉 Visit this URL in your browser and authorize:\n")
print(authorize_url)


## 4) Exchange Code for Token

In [ ]:
# Paste code from browser
auth_code = input("\nPaste the authorization code: ").strip()

# Exchange for tokens
token_response = client.oauth2_exchange_code_for_tokens(auth_code)

fs_token = None

# Extract filesystem token
for rs, token_data in token_response.by_resource_server.items():
    if "filesystem" in token_data.get("scope", ""):
        fs_token = token_data["access_token"]
        expires_at = token_data["expires_at_seconds"]

        expiration_time = datetime.datetime.fromtimestamp(expires_at)
        print(f"\n📦 Filesystem token expires at: {expiration_time}")

        seconds_left = expires_at - time.time()
        hours_left = seconds_left / 3600
        print(f"Token expires in {hours_left:.2f} hours")

        print(f"\n=== USE THIS TOKEN for ACCESING ALCF ===\n")
        print(fs_token)

        break

if not fs_token:
    print("\nERROR: No filesystem token found. Check scope.")

## 5) Save Token for Other Notebooks

In [ ]:
home_path = Path("~/.iri_token.json").expanduser()

home_path.parent.mkdir(parents=True, exist_ok=True)

with open(home_path, "w") as f:
    json.dump({"IRI_API_TOKEN": iri_token}, f)
print(f"Token saved: {home_path}")